# EDA Notebook: Offline Demo Path for Fitness Supplements Reviews

This notebook is a small exploratory deliverable for Assignment 1. It is intentionally aligned with the current offline demo path and any local collected artifacts that may already exist in the repository. It does **not** claim to represent a production corpus. If the local artifacts are unavailable, the notebook falls back to a small demo-aligned dataframe so the analysis remains runnable offline.

## Dataset Overview

The first step is to look for local, offline-friendly review data. The notebook tries project artifacts first, then falls back to a small demo dataframe only when necessary.

Expected outputs in this section: number of rows, columns, and a few sample records.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("ggplot")

NOTEBOOK_ROOT = Path.cwd().resolve()
CANDIDATE_DATA_PATHS = [
    NOTEBOOK_ROOT / "data" / "interim" / "review_queue_corrected.csv",
    NOTEBOOK_ROOT / "data" / "interim" / "review_queue.csv",
    NOTEBOOK_ROOT / "tests" / "fixtures" / "sample_review_queue.csv",
]
STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "have", "has", "was",
    "were", "are", "you", "your", "our", "but", "not", "too", "very", "about",
    "review", "reviews", "supplement", "supplements", "fitness", "product", "products",
    "good", "bad", "great", "nice", "one", "use", "using", "made", "make", "can",
    "would", "could", "just", "really", "more", "less", "after", "before", "also",
}

def load_offline_eda_frame() -> tuple[pd.DataFrame, str]:
    """Load a local review dataset or fall back to a small demo-aligned dataframe."""

    for path in CANDIDATE_DATA_PATHS:
        if not path.exists():
            continue
        try:
            frame = pd.read_csv(path)
            return frame, f"local artifact: {path.relative_to(NOTEBOOK_ROOT)}"
        except Exception:
            continue

    demo_rows = [
        {
            "id": "1",
            "source": "offline_demo",
            "text": "Great energy boost for workouts and long runs",
            "label": None,
            "rating": 5,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "positive",
            "effect_label": "energy",
            "confidence": 0.96,
            "reviewed_effect_label": "energy",
            "review_comment": "",
            "human_verified": True,
        },
        {
            "id": "2",
            "source": "offline_demo",
            "text": "Side effect warning after taking the powder at night",
            "label": None,
            "rating": 1,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "negative",
            "effect_label": "side_effects",
            "confidence": 0.81,
            "reviewed_effect_label": "side_effects",
            "review_comment": "",
            "human_verified": True,
        },
        {
            "id": "3",
            "source": "offline_demo",
            "text": "Neutral supplement routine with no strong effect",
            "label": None,
            "rating": 3,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "neutral",
            "effect_label": "other",
            "confidence": 0.58,
            "reviewed_effect_label": "other",
            "review_comment": "",
            "human_verified": False,
        },
        {
            "id": "4",
            "source": "offline_demo",
            "text": "Energy and focus felt stronger during training sessions",
            "label": None,
            "rating": 5,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "positive",
            "effect_label": "energy",
            "confidence": 0.91,
            "reviewed_effect_label": "energy",
            "review_comment": "",
            "human_verified": True,
        },
        {
            "id": "5",
            "source": "offline_demo",
            "text": "A bit of stomach discomfort after the first dose",
            "label": None,
            "rating": 2,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "negative",
            "effect_label": "side_effects",
            "confidence": 0.74,
            "reviewed_effect_label": "side_effects",
            "review_comment": "",
            "human_verified": False,
        },
        {
            "id": "6",
            "source": "offline_demo",
            "text": "Balanced routine with mixed results and no clear side effects",
            "label": None,
            "rating": 3,
            "created_at": "2026-03-27",
            "split": None,
            "meta_json": "{}",
            "sentiment_label": "neutral",
            "effect_label": "other",
            "confidence": 0.66,
            "reviewed_effect_label": "other",
            "review_comment": "",
            "human_verified": False,
        },
    ]
    return pd.DataFrame(demo_rows), "inline demo-aligned dataframe"

def choose_text_column(frame: pd.DataFrame) -> str:
    """Select the best available text column for EDA."""

    for candidate in ("text", "content", "review_text", "body"):
        if candidate in frame.columns:
            return candidate
    return frame.columns[0]

def choose_label_column(frame: pd.DataFrame) -> str | None:
    """Select a label column with a truthful fallback order."""

    for candidate in ("effect_label", "reviewed_effect_label", "label", "sentiment_label"):
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None

def tokenize(text: str) -> list[str]:
    """Tokenize text with a small regex-based approach and a short stopword filter."""

    raw_tokens = re.findall(r"[a-zA-Z']+", text.lower())
    return [token for token in raw_tokens if len(token) > 2 and token not in STOPWORDS]

df, source_note = load_offline_eda_frame()
text_column = choose_text_column(df)
label_column = choose_label_column(df)

print(f"Data source: {source_note}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(5)

## Schema Overview

This section shows the current column set, dtypes, and a quick non-null snapshot. It is intentionally lightweight so it matches the offline demo baseline and stays easy to inspect.

In [ ]:
schema_overview = pd.DataFrame({
    "column": list(df.columns),
    "dtype": [str(dtype) for dtype in df.dtypes],
    "non_null": [int(df[column].notna().sum()) for column in df.columns],
    "example_value": [df[column].dropna().iloc[0] if df[column].notna().any() else None for column in df.columns],
})
schema_overview

## Text Length Analysis

The goal here is to understand the rough shape of the review texts before cleaning or modeling. Short, review-style snippets usually benefit from light normalization and consistent token filtering.

In [ ]:
text_series = df[text_column].fillna("").astype(str)
char_lengths = text_series.map(len)
word_lengths = text_series.map(lambda value: len(tokenize(value)))
length_stats = pd.DataFrame({
    "chars": char_lengths.describe(),
    "words": word_lengths.describe(),
})
length_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(char_lengths, bins=min(12, max(1, len(char_lengths))))
axes[0].set_title("Text length in characters")
axes[0].set_xlabel("characters")
axes[0].set_ylabel("rows")
axes[1].hist(word_lengths, bins=min(12, max(1, len(word_lengths))))
axes[1].set_title("Text length in words")
axes[1].set_xlabel("words")
axes[1].set_ylabel("rows")
fig.tight_layout()
plt.show()

## Label Distribution

If `effect_label` is available, it is used as the primary label field because it matches the current text -> effect_label baseline. Otherwise, the notebook falls back to another present label-like field and calls that out explicitly.

In [ ]:
if label_column is None:
    print("No usable label column was found in the loaded frame.")
    label_counts = pd.Series(dtype="int64")
else:
    label_counts = (
        df[label_column]
        .fillna("missing")
        .astype(str)
        .str.strip()
        .replace("", "missing")
        .value_counts()
    )
    print(f"Label column: {label_column}")
    display(label_counts.to_frame(name="count"))

In [ ]:
if not label_counts.empty:
    ax = label_counts.head(20).plot(kind="bar", figsize=(10, 4), color="steelblue")
    ax.set_title("Label distribution")
    ax.set_xlabel("label")
    ax.set_ylabel("count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Top-20 Words

A small regex tokenizer is enough for this deliverable. The intent is not to over-engineer NLP preprocessing here, but to get a readable sense of the most common terms in the offline demo path.

In [ ]:
all_tokens: list[str] = []
for value in text_series:
    all_tokens.extend(tokenize(value))

top_words = pd.DataFrame(Counter(all_tokens).most_common(20), columns=["word", "count"])
top_words

In [ ]:
if not top_words.empty:
    ax = top_words.plot(kind="bar", x="word", y="count", legend=False, figsize=(10, 4), color="darkorange")
    ax.set_title("Top-20 words")
    ax.set_xlabel("word")
    ax.set_ylabel("count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Short Conclusions / Limitations

- This notebook analyzes the current offline demo path or a demo-aligned fallback frame, not a production corpus.
- The review texts are short and domain-specific, so simple cleaning and token filtering are enough for the assignment-level EDA.
- If the label distribution is skewed, downstream annotation and training should keep using stratified or balanced handling where possible.
- Common words are likely dominated by review boilerplate, so later preprocessing can focus on consistent normalization rather than aggressive NLP tooling.
- The notebook stays intentionally lightweight so it remains easy to run offline and easy to compare against future real data without changing the baseline pipeline.